# MCP — resources and prompts

Tools execute actions. **Resources** are addressable read-only context (URIs like `arxiv://synth-001`). **Prompts** are server-owned templates with declared arguments. Together they let an MCP server expose a complete agent application surface — not just a tool palette.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / '06-mcp'))
os.environ.setdefault('LLM_CACHE_ONLY', '1')

from mcp_core import build_corpus_server, Client, InProcessTransport

server = build_corpus_server(with_resources=True, with_prompts=True)
client = Client(transport=InProcessTransport(server)); client.initialize()
[r['uri'] for r in client.list_resources()][:5]

In [ ]:
# Read a single resource — like an HTTP GET on a stable URI.
print(client.read_resource('arxiv://synth-001')[:300])

In [ ]:
for p in client.list_prompts():
    print(p['name'], '|', p['description'], '|', p['arguments'])

messages = client.get_prompt('research', {'question': 'Summarise RA-MoE.'})
for m in messages:
    print(m['role'], '->', m['content']['text'][:120])

## Why this matters

* The prompt template lives **on the server**, so clients can't drift.
* Resources have **URIs**, so caching, audit, and access control work the way they do for HTTP.
* `resources/list` lets the LLM *discover* what's available — much better than a static system-prompt enumeration.